# Downloading ECCO Data

Author: Mike Wood

This notebook is designed to download Landsat data from Earth Explorer.

Note that you must have an [Earth Explorer](https://earthexplorer.usgs.gov/) account. If you don't already have one, be sure to make one before using this notebook.


#### Import the modules for this notebook

In [ ]:
import os
from landsatxplore.api import API
from landsatxplore.earthexplorer import EarthExplorer
import getpass
import tarfile

#### Define the destination file path

In [ ]:
# define the path to the folder where the data will be downloaded
download_folder = ''

#### Define the data request

In [ ]:
# choose an a Landsat number
landsat_number = '8' #5, 7, or 8

# choose a level
level = '2' #either 1 or 2

# choose a location
longitude = -121.9
latitude = 36.8

# choose a start date and end date
start_date = '2016-01-01'
end_date = '2017-01-01'

# specify max cloud cover (choose 100 for all scenes)
max_cloud_cover = 50

Run this cell to format the data set:

In [ ]:
landsat_key = {'5':'tm', '7':'etm', '8':'ot'}
dataset = 'landsat_'+landsat_key[landsat_number]+'_c2_l'+str(level)

#### Make a list of file paths
Nothing to do here - just run this cell to generate the list.

In [ ]:
print('Enter your Earth Explorer credentials')
username = getpass.getpass('Enter your username: ')
password = getpass.getpass('Enter your password: ')

# Initialize a new API instance and get an access key
api = API(username, password)

# Search for Landsat TM scenes
scenes = api.search(
    dataset=dataset,
    latitude=latitude,
    longitude=longitude,
    start_date=start_date,
    end_date=end_date,
    max_cloud_cover=max_cloud_cover
)

print(f"{len(scenes)} scenes found.")

api.logout()

#### Download the data

In [ ]:
ee = EarthExplorer(username, password)
for s in range(len(scenes)):
    scene = scenes[s]
    if scene['landsat_product_id']+'.tar' not in os.listdir(download_folder):
        print('Downloading scene '+scene['entity_id']+' ('+str(s+1)+' of '+str(len(scenes))+')')
        ee.download(scene['entity_id'], output_dir=download_folder)
ee.logout()

### Untar Scenes (if desired)
The Landsat scenes from Earth Explorer are delivered as a .tar file. If you would like to extract some or all of the files from the downloaded scenes, then run the following cell:

In [ ]:
# choose the bands to extract
bands = [10, 11] 

# untar the scenes
for s in range(len(scenes)):
    scene = scenes[s]
    if scene['landsat_product_id']+'.tar' in os.listdir(download_folder):

        # make a folder for the extracted files
        if scene['landsat_product_id'] not in os.listdir(download_folder):
            os.mkdir(os.path.join(download_folder, scene['landsat_product_id']))

        # extract the files
        tar = tarfile.open(os.path.join(download_folder,scene['landsat_product_id']+'.tar'))
        for band in bands:
            tar.extract(scene['landsat_product_id']+'_B'+str(band)+'.TIF',
                os.path.join(download_folder,scene['landsat_product_id']))
        tar.extract(scene['landsat_product_id']+'_MTL.txt',
            os.path.join(download_folder,scene['landsat_product_id']))
        tar.close()